In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import dados, pivot, salvar
from outliers import detect_outliers_iqr, detect_outliers_lof, detect_outliers_pca_kmeans, detect_outliers_zscore, detect_outliers

In [ ]:
def remove_outliers_zscore(df, variaveis, threshold=3, salvar_arquivo=True):
    """
    Remove outliers de acordo com o método zscore
    """
    _, _, outliers_zscore = detect_outliers_zscore(df, variaveis, threshold)
    keep = ~outliers_zscore.any(axis=1)
    df_clean = df.loc[keep].copy()
    
    if salvar_arquivo:
        salvar(df_clean, "outliers_removed_zscore")
    
    return df_clean

In [ ]:
def remove_outliers_iqr(df, variaveis, salvar_arquivo=True):
    """
    Remove outliers de acordo com o método IQR
    """
    _, _, outliers_por_linha_iqr = detect_outliers_iqr(df, variaveis)
    df_clean = df.loc[~outliers_por_linha_iqr].copy()
    
    if salvar_arquivo:
        salvar(df_clean, "outliers_removed_iqr")
    
    return df_clean

In [ ]:
def remove_outliers_lof(df, variaveis, n_neighbors=20, contamination='auto', salvar_arquivo=True):
    """
    Remove outliers de acordo com o método LOF
    """
    _, outlier_labels, _ = detect_outliers_lof(df, variaveis, n_neighbors, contamination)
    df_clean = df.loc[outlier_labels == 1].copy()
    
    if salvar_arquivo:
        salvar(df_clean, "outliers_removed_lof")
    
    return df_clean

In [ ]:
def remove_outliers(df, variaveis, method='zscore', salvar_arquivo=True, **kwargs):
    """
    Remove outliers com o método especificado.
    
    df: DataFrame.
    variaveis: Lista de colunas numéricas.
    method: 'zscore' (default), 'iqr', ou 'lof'.
    salvar_arquivo: Salva o arquivo (default: Ativado).
    **kwargs: Parâmetros adicionais para o método (threshold, n_neighbors, etc.).
    """
    if method == 'zscore':
        return remove_outliers_zscore(df, variaveis, salvar_arquivo=salvar_arquivo, **kwargs)
    elif method == 'iqr':
        return remove_outliers_iqr(df, variaveis, salvar_arquivo=salvar_arquivo)
    elif method == 'lof':
        return remove_outliers_lof(df, variaveis, salvar_arquivo=salvar_arquivo, **kwargs)
    else:
        raise ValueError(f"Unknown method: {method}")

In [3]:
dados_limpos, metadados, variaveis = dados(r"data\out\dados_limpos20260410_165530.xlsx")
remove_outliers_iqr(dados_limpos, variaveis=variaveis)

Outliers detectados por IQR (Tukey):
COR: 55 outliers (limites: -141.500 a 358.500)
TURB.: 75 outliers (limites: -115.500 a 216.500)
pH: 21 outliers (limites: 6.750 a 7.950)
ALC.: 28 outliers (limites: -3.625 a 123.375)
AC.: 51 outliers (limites: -2.000 a 22.000)
O.C.: 37 outliers (limites: 0.450 a 16.050)
O.D.: 5 outliers (limites: -1.200 a 7.600)
Cl: 38 outliers (limites: -1.500 a 82.500)
DUR.: 23 outliers (limites: 4.000 a 100.000)
Fe: 53 outliers (limites: -2.163 a 5.818)
Mn: 60 outliers (limites: -0.055 a 0.385)
Cond.: 25 outliers (limites: -98.250 a 703.750)
Cianobacteria: 46 outliers (limites: -14886.000 a 30314.000)
C.F.: 57 outliers (limites: -23000.000 a 44200.000)
Clorofila: 33 outliers (limites: -40.055 a 99.505)
F: 26 outliers (limites: -0.076 a 0.654)

Total de outliers (IQR, somando todas as colunas): 633
Total de linhas com pelo menos um outlier (IQR): 269


,Index,Data,data_normalizada,Estatistica,COR,TURB.,pH,ALC.,AC.,O.C.,O.D.,Cl,DUR.,Fe,Mn,Cond.,Cianobacteria,C.F.,Clorofila,F
0,0,2009-01-01,2009-01-01,Min.,200.0,64.0,7.0,29.0,3.0,6.3,2.0,16.0,36.0,3.23,0.09,131.0,6600.0,23000.0,2.23,0.12
9,9,2009-04-01,2009-04-01,Min.,40.0,15.0,7.3,44.0,6.0,6.8,1.8,20.0,40.0,1.35,0.10,176.0,2525.0,17000.0,8.03,0.12
10,10,2009-04-01,2009-04-01,Med.,63.0,36.0,7.4,47.0,10.0,7.4,2.7,23.0,45.0,1.96,0.12,201.0,6920.0,17000.0,9.93,0.16
11,11,2009-04-01,2009-04-01,Max.,90.0,47.0,7.5,49.0,12.0,7.8,3.5,25.0,50.0,2.49,0.12,239.0,11314.0,17000.0,12.05,0.19
12,12,2009-05-01,2009-05-01,Min.,23.0,12.0,7.1,51.0,4.0,4.0,1.4,30.0,36.0,1.04,0.11,281.0,4348.0,1100.0,8.03,0.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,570,2024-11-01,2024-11-01,Min.,72.0,10.0,7.0,41.0,5.0,5.2,1.9,31.0,38.0,0.68,0.03,150.0,NaN,3000.0,38.68,0.24
571,571,2024-11-01,2024-11-01,Med.,217.0,134.0,7.5,48.0,9.0,9.7,4.2,39.0,63.0,1.57,0.11,263.0,NaN,3000.0,38.68,0.32
573,573,2024-12-01,2024-12-01,Min.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
574,574,2024-12-01,2024-12-01,Med.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
